# تشخیص و تحلیل کانتور / Contour Detection and Analysis

**دانشجو / Student:** مسیح معافی / Masih Moafi  
**تاریخ / Date:** ۱۴۰۳/۰۷/۲۸ (October 19, 2025)

---

## هدف / Objective

**هدف:** یادگیری یافتن و تحلیل کانتورها در تصاویر

**Objective:** Learn to find and analyze contours in images

---

## محتوا / Contents

1. مقدمه‌ای بر کانتورها / Introduction to Contours
2. یافتن کانتورها / Finding Contours
3. رسم کانتورها / Drawing Contours
4. ویژگی‌های کانتور / Contour Properties
5. تقریب کانتور / Contour Approximation
6. پوسته محدب / Convex Hull
7. تطبیق شکل / Shape Matching
8. سلسله‌مراتب کانتورها / Contour Hierarchy

In [ ]:
# وارد کردن کتابخانه‌های مورد نیاز / Import required libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List
import os

# تنظیمات نمایش / Display settings
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 12

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. توابع کمکی / Helper Functions

In [ ]:
def display_images(images: list, titles: list, cmap='gray', rows=1):
    """
    نمایش چند تصویر در کنار هم
    """
    n = len(images)
    cols = (n + rows - 1) // rows
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    
    if n == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if rows > 1 or cols > 1 else [axes]
    
    for i, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 3:
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            axes[i].imshow(img, cmap=cmap)
        axes[i].set_title(title, fontsize=14, fontweight='bold')
        axes[i].axis('off')
    
    for i in range(n, len(axes)):
        axes[i].remove()
    
    plt.tight_layout()
    plt.show()

print("✓ توابع کمکی آماده شدند")

## 2. ایجاد تصویر نمونه / Create Sample Image

In [ ]:
# ایجاد تصویر با اشکال مختلف
# Create image with different shapes
image = np.zeros((500, 700, 3), dtype=np.uint8)

# اضافه کردن اشکال
# Add shapes
cv2.rectangle(image, (50, 50), (150, 150), (255, 0, 0), -1)  # مربع آبی / Blue square
cv2.circle(image, (250, 100), 50, (0, 255, 0), -1)  # دایره سبز / Green circle
cv2.ellipse(image, (400, 100), (60, 40), 0, 0, 360, (0, 0, 255), -1)  # بیضی قرمز / Red ellipse

# مثلث / Triangle
triangle = np.array([[550, 50], [600, 150], [500, 150]], np.int32)
cv2.fillPoly(image, [triangle], (255, 255, 0))

# چندضلعی / Polygon
pentagon = np.array([[100, 300], [150, 250], [200, 270], [190, 330], [110, 340]], np.int32)
cv2.fillPoly(image, [pentagon], (255, 0, 255))

# ستاره / Star
star = np.array([[400, 250], [420, 300], [470, 310], [430, 340], [440, 390],
                 [400, 360], [360, 390], [370, 340], [330, 310], [380, 300]], np.int32)
cv2.fillPoly(image, [star], (0, 255, 255))

# تبدیل به خاکستری
# Convert to grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# آستانه‌گذاری
# Thresholding
_, binary = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)

display_images([image, gray, binary],
               ['تصویر رنگی\nColor Image', 'خاکستری\nGrayscale', 'باینری\nBinary'])

print(f"اندازه تصویر / Image size: {image.shape}")

## 3. یافتن کانتورها / Finding Contours

تابع `cv2.findContours()` برای یافتن کانتورها استفاده می‌شود.

The `cv2.findContours()` function is used to find contours.

In [ ]:
# یافتن کانتورها
# Find contours
contours, hierarchy = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

print(f"تعداد کانتورها / Number of contours: {len(contours)}")
print(f"\nاطلاعات کانتورها / Contour information:")
for i, contour in enumerate(contours):
    print(f"کانتور {i}: {len(contour)} نقطه / Contour {i}: {len(contour)} points")

## 4. رسم کانتورها / Drawing Contours

In [ ]:
# رسم همه کانتورها
# Draw all contours
image_all = image.copy()
cv2.drawContours(image_all, contours, -1, (0, 255, 0), 2)

# رسم کانتورهای جداگانه با رنگ‌های مختلف
# Draw individual contours with different colors
image_colored = image.copy()
colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255), (0, 255, 255)]

for i, contour in enumerate(contours):
    color = colors[i % len(colors)]
    cv2.drawContours(image_colored, [contour], -1, color, 3)

display_images([image, image_all, image_colored],
               ['اصلی\nOriginal', 'همه کانتورها\nAll Contours', 'کانتورهای رنگی\nColored Contours'])

print("✓ کانتورها رسم شدند")

### حالت‌های مختلف بازیابی / Different Retrieval Modes

In [ ]:
# ایجاد تصویر با اشکال تو در تو
# Create image with nested shapes
nested_image = np.zeros((400, 400), dtype=np.uint8)
cv2.rectangle(nested_image, (50, 50), (350, 350), 255, -1)
cv2.rectangle(nested_image, (100, 100), (300, 300), 0, -1)
cv2.rectangle(nested_image, (150, 150), (250, 250), 255, -1)

# حالت‌های مختلف بازیابی
# Different retrieval modes
modes = [
    (cv2.RETR_EXTERNAL, 'EXTERNAL'),
    (cv2.RETR_LIST, 'LIST'),
    (cv2.RETR_TREE, 'TREE')
]

results = []
for mode, name in modes:
    contours_mode, _ = cv2.findContours(nested_image, mode, cv2.CHAIN_APPROX_SIMPLE)
    img_result = cv2.cvtColor(nested_image, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(img_result, contours_mode, -1, (0, 255, 0), 2)
    results.append(img_result)
    print(f"{name}: {len(contours_mode)} کانتور / contours")

images = [cv2.cvtColor(nested_image, cv2.COLOR_GRAY2BGR)] + results
titles = ['اصلی\nOriginal', 'EXTERNAL', 'LIST', 'TREE']
display_images(images, titles, rows=2)

## 5. ویژگی‌های کانتور / Contour Properties

In [ ]:
# محاسبه ویژگی‌های کانتورها
# Calculate contour properties
image_props = image.copy()

print("\nویژگی‌های کانتورها / Contour Properties:\n")

for i, contour in enumerate(contours):
    # مساحت / Area
    area = cv2.contourArea(contour)
    
    # محیط / Perimeter
    perimeter = cv2.arcLength(contour, True)
    
    # مرکز جرم / Centroid
    M = cv2.moments(contour)
    if M['m00'] != 0:
        cx = int(M['m10'] / M['m00'])
        cy = int(M['m01'] / M['m00'])
    else:
        cx, cy = 0, 0
    
    # مستطیل محیطی / Bounding rectangle
    x, y, w, h = cv2.boundingRect(contour)
    
    # رسم مستطیل محیطی و مرکز
    # Draw bounding rectangle and center
    cv2.rectangle(image_props, (x, y), (x+w, y+h), (0, 255, 0), 2)
    cv2.circle(image_props, (cx, cy), 5, (0, 0, 255), -1)
    cv2.putText(image_props, str(i), (cx-10, cy-10), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    
    print(f"کانتور {i} / Contour {i}:")
    print(f"  مساحت / Area: {area:.2f}")
    print(f"  محیط / Perimeter: {perimeter:.2f}")
    print(f"  مرکز / Center: ({cx}, {cy})")
    print(f"  مستطیل / Bounding box: ({x}, {y}, {w}, {h})")
    print()

display_images([image, image_props],
               ['اصلی\nOriginal', 'ویژگی‌ها\nProperties'])

print("✓ ویژگی‌های کانتور محاسبه شد")

## 6. تقریب کانتور / Contour Approximation

تقریب کانتور با استفاده از الگوریتم Douglas-Peucker

Contour approximation using Douglas-Peucker algorithm

In [ ]:
# تقریب کانتورها با ضرایب مختلف
# Approximate contours with different epsilons
epsilons = [0.01, 0.02, 0.05]

for epsilon_factor in epsilons:
    image_approx = image.copy()
    
    for contour in contours:
        # محاسبه epsilon
        # Calculate epsilon
        epsilon = epsilon_factor * cv2.arcLength(contour, True)
        
        # تقریب کانتور
        # Approximate contour
        approx = cv2.approxPolyDP(contour, epsilon, True)
        
        # رسم کانتور تقریبی
        # Draw approximated contour
        cv2.drawContours(image_approx, [approx], -1, (0, 255, 0), 2)
        
        # رسم نقاط گوشه
        # Draw corner points
        for point in approx:
            cv2.circle(image_approx, tuple(point[0]), 5, (0, 0, 255), -1)
    
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(image_approx, cv2.COLOR_BGR2RGB))
    plt.title(f'تقریب با ε={epsilon_factor}\nApproximation with ε={epsilon_factor}', 
              fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

print("✓ تقریب کانتور انجام شد")

### تشخیص شکل بر اساس تعداد گوشه‌ها / Shape Detection Based on Corners

In [ ]:
def detect_shape(contour):
    """
    تشخیص شکل بر اساس تعداد گوشه‌ها
    
    Args:
        contour: کانتور
    
    Returns:
        نام شکل
    """
    # تقریب کانتور
    # Approximate contour
    epsilon = 0.04 * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, epsilon, True)
    
    # تعداد گوشه‌ها
    # Number of corners
    vertices = len(approx)
    
    if vertices == 3:
        return "مثلث / Triangle"
    elif vertices == 4:
        # بررسی مربع یا مستطیل
        # Check square or rectangle
        x, y, w, h = cv2.boundingRect(approx)
        aspect_ratio = float(w) / h
        if 0.95 <= aspect_ratio <= 1.05:
            return "مربع / Square"
        else:
            return "مستطیل / Rectangle"
    elif vertices == 5:
        return "پنج‌ضلعی / Pentagon"
    elif vertices > 5 and vertices < 10:
        return f"{vertices}-ضلعی / {vertices}-gon"
    else:
        # بررسی دایره
        # Check circle
        area = cv2.contourArea(contour)
        perimeter = cv2.arcLength(contour, True)
        if perimeter > 0:
            circularity = 4 * np.pi * area / (perimeter ** 2)
            if circularity > 0.8:
                return "دایره / Circle"
        return "شکل پیچیده / Complex shape"


# تشخیص و برچسب‌گذاری اشکال
# Detect and label shapes
image_shapes = image.copy()

for i, contour in enumerate(contours):
    shape = detect_shape(contour)
    
    # محاسبه مرکز
    # Calculate center
    M = cv2.moments(contour)
    if M['m00'] != 0:
        cx = int(M['m10'] / M['m00'])
        cy = int(M['m01'] / M['m00'])
        
        # نوشتن نام شکل
        # Write shape name
        cv2.putText(image_shapes, shape.split('/')[0].strip(), (cx-30, cy), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    
    # رسم کانتور
    # Draw contour
    cv2.drawContours(image_shapes, [contour], -1, (0, 255, 0), 2)
    
    print(f"کانتور {i}: {shape}")

display_images([image, image_shapes],
               ['اصلی\nOriginal', 'تشخیص شکل\nShape Detection'])

print("\n✓ اشکال تشخیص داده شدند")

## 7. پوسته محدب / Convex Hull

In [ ]:
# محاسبه پوسته محدب
# Calculate convex hull
image_hull = image.copy()

for contour in contours:
    # محاسبه پوسته محدب
    # Calculate convex hull
    hull = cv2.convexHull(contour)
    
    # رسم کانتور اصلی
    # Draw original contour
    cv2.drawContours(image_hull, [contour], -1, (0, 255, 0), 2)
    
    # رسم پوسته محدب
    # Draw convex hull
    cv2.drawContours(image_hull, [hull], -1, (0, 0, 255), 2)

display_images([image, image_hull],
               ['اصلی\nOriginal', 'پوسته محدب (قرمز)\nConvex Hull (red)'])

print("✓ پوسته محدب محاسبه شد")

### بررسی محدب بودن / Convexity Check

In [ ]:
# بررسی محدب بودن کانتورها
# Check convexity of contours
print("بررسی محدب بودن / Convexity Check:\n")

for i, contour in enumerate(contours):
    is_convex = cv2.isContourConvex(contour)
    
    # محاسبه نقص محدب
    # Calculate convexity defects
    hull = cv2.convexHull(contour, returnPoints=False)
    
    if len(hull) > 3 and len(contour) > 3:
        defects = cv2.convexityDefects(contour, hull)
        if defects is not None:
            num_defects = len(defects)
        else:
            num_defects = 0
    else:
        num_defects = 0
    
    print(f"کانتور {i}: محدب={is_convex}, نقص‌ها={num_defects}")
    print(f"Contour {i}: convex={is_convex}, defects={num_defects}\n")

## 8. تطبیق شکل / Shape Matching

In [ ]:
# ایجاد شکل مرجع (مثلث)
# Create reference shape (triangle)
ref_image = np.zeros((200, 200), dtype=np.uint8)
triangle_ref = np.array([[100, 30], [30, 170], [170, 170]], np.int32)
cv2.fillPoly(ref_image, [triangle_ref], 255)

# یافتن کانتور مرجع
# Find reference contour
ref_contours, _ = cv2.findContours(ref_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
ref_contour = ref_contours[0]

# مقایسه با کانتورهای تصویر اصلی
# Compare with original image contours
print("تطبیق شکل / Shape Matching:\n")

for i, contour in enumerate(contours):
    # محاسبه شباهت
    # Calculate similarity
    match = cv2.matchShapes(ref_contour, contour, cv2.CONTOURS_MATCH_I1, 0)
    
    print(f"کانتور {i}: شباهت با مثلث = {match:.4f}")
    print(f"Contour {i}: similarity to triangle = {match:.4f}\n")

# نمایش شکل مرجع
# Display reference shape
plt.figure(figsize=(5, 5))
plt.imshow(ref_image, cmap='gray')
plt.title('شکل مرجع (مثلث)\nReference Shape (Triangle)', fontweight='bold')
plt.axis('off')
plt.show()

print("✓ تطبیق شکل انجام شد")
print("نکته: مقدار کمتر = شباهت بیشتر / Note: lower value = more similar")

## 9. سلسله‌مراتب کانتورها / Contour Hierarchy

In [ ]:
# ایجاد تصویر با کانتورهای تو در تو
# Create image with nested contours
hierarchy_image = np.zeros((400, 600), dtype=np.uint8)

# مستطیل بیرونی
# Outer rectangle
cv2.rectangle(hierarchy_image, (50, 50), (550, 350), 255, -1)

# مستطیل میانی (سوراخ)
# Middle rectangle (hole)
cv2.rectangle(hierarchy_image, (100, 100), (500, 300), 0, -1)

# مستطیل داخلی
# Inner rectangle
cv2.rectangle(hierarchy_image, (200, 150), (400, 250), 255, -1)

# دایره داخلی
# Inner circle
cv2.circle(hierarchy_image, (300, 200), 30, 0, -1)

# یافتن کانتورها با سلسله‌مراتب
# Find contours with hierarchy
contours_hier, hierarchy_info = cv2.findContours(hierarchy_image, cv2.RETR_TREE, 
                                                  cv2.CHAIN_APPROX_SIMPLE)

# نمایش اطلاعات سلسله‌مراتب
# Display hierarchy information
print("سلسله‌مراتب کانتورها / Contour Hierarchy:\n")
print("فرمت: [بعدی، قبلی، اولین فرزند، والد]")
print("Format: [Next, Previous, First Child, Parent]\n")

for i, h in enumerate(hierarchy_info[0]):
    print(f"کانتور {i}: {h}")

# رسم کانتورها با رنگ‌های مختلف بر اساس سطح
# Draw contours with different colors based on level
hierarchy_colored = cv2.cvtColor(hierarchy_image, cv2.COLOR_GRAY2BGR)
colors_hier = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0)]

for i, contour in enumerate(contours_hier):
    color = colors_hier[i % len(colors_hier)]
    cv2.drawContours(hierarchy_colored, [contour], -1, color, 3)
    
    # نوشتن شماره کانتور
    # Write contour number
    M = cv2.moments(contour)
    if M['m00'] != 0:
        cx = int(M['m10'] / M['m00'])
        cy = int(M['m01'] / M['m00'])
        cv2.putText(hierarchy_colored, str(i), (cx-10, cy), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

display_images([cv2.cvtColor(hierarchy_image, cv2.COLOR_GRAY2BGR), hierarchy_colored],
               ['اصلی\nOriginal', 'سلسله‌مراتب\nHierarchy'])

print("\n✓ سلسله‌مراتب کانتورها نمایش داده شد")

## 10. کاربرد عملی: شمارش اشیاء / Practical Application: Object Counting

In [ ]:
# ایجاد تصویر با اشیاء متعدد
# Create image with multiple objects
counting_image = np.zeros((400, 600), dtype=np.uint8)

# اضافه کردن دایره‌های تصادفی
# Add random circles
np.random.seed(42)
num_objects = 15

for _ in range(num_objects):
    x = np.random.randint(50, 550)
    y = np.random.randint(50, 350)
    radius = np.random.randint(15, 35)
    cv2.circle(counting_image, (x, y), radius, 255, -1)

# یافتن و شمارش کانتورها
# Find and count contours
contours_count, _ = cv2.findContours(counting_image, cv2.RETR_EXTERNAL, 
                                      cv2.CHAIN_APPROX_SIMPLE)

# فیلتر کردن کانتورهای کوچک
# Filter small contours
min_area = 200
valid_contours = [c for c in contours_count if cv2.contourArea(c) > min_area]

# رسم نتایج
# Draw results
counting_result = cv2.cvtColor(counting_image, cv2.COLOR_GRAY2BGR)

for i, contour in enumerate(valid_contours):
    # رسم کانتور
    # Draw contour
    cv2.drawContours(counting_result, [contour], -1, (0, 255, 0), 2)
    
    # شماره‌گذاری
    # Numbering
    M = cv2.moments(contour)
    if M['m00'] != 0:
        cx = int(M['m10'] / M['m00'])
        cy = int(M['m01'] / M['m00'])
        cv2.putText(counting_result, str(i+1), (cx-10, cy+5), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

# نوشتن تعداد کل
# Write total count
cv2.putText(counting_result, f'Total: {len(valid_contours)}', (10, 30), 
            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

display_images([cv2.cvtColor(counting_image, cv2.COLOR_GRAY2BGR), counting_result],
               ['اصلی\nOriginal', f'شمارش: {len(valid_contours)}\nCount: {len(valid_contours)}'])

print(f"✓ تعداد اشیاء شمارش شد: {len(valid_contours)}")
print(f"✓ Objects counted: {len(valid_contours)}")

## 11. تمرین‌ها / Exercises

### تمرین 1 / Exercise 1
تصویری با اشکال مختلف ایجاد کنید و برنامه‌ای بنویسید که هر شکل را تشخیص دهد.

Create an image with different shapes and write a program that detects each shape.

### تمرین 2 / Exercise 2
برنامه‌ای بنویسید که بزرگ‌ترین و کوچک‌ترین کانتور را پیدا کند.

Write a program that finds the largest and smallest contour.

### تمرین 3 / Exercise 3
کانتورهای یک تصویر را بر اساس مساحت مرتب کنید.

Sort the contours of an image based on area.

### تمرین 4 / Exercise 4
برنامه‌ای بنویسید که فقط کانتورهای محدب را نمایش دهد.

Write a program that displays only convex contours.

### تمرین 5 / Exercise 5
از تطبیق شکل برای یافتن یک شکل خاص در تصویر استفاده کنید.

Use shape matching to find a specific shape in an image.